# Maximal Marginal Relevance (MMR) Retrieval in Production RAG Pipelines
### Architectural Reference Implementation for Enterprise Systems

This reference notebook implements an end-to-end Retrieval-Augmented Generation (RAG) pipeline leveraging Maximal Marginal Relevance (MMR) for search diversification. When querying vector databases, standard dense retrieval frequently returns redundant document chunks discussing the same subset of ideas. MMR dynamically balances relevance to the user query against inter-document novelty, suppressing redundant candidates and expanding contextual coverage for downstream LLM synthesis.

## 1. Environment Configuration & Library Imports

We import standard LangChain components: `TextLoader`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, `FAISS`, and LCEL retrieval chain factories.

In [1]:
import os
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()
print("Dependencies loaded successfully.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_7104\2522160290.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Dependencies loaded successfully.


## 2. Document Ingestion & Chunking

We load `langchain_rag_dataset.txt` and segment it using `RecursiveCharacterTextSplitter` configured with `chunk_size=300` and `chunk_overlap=50`.

In [2]:
data_path: str = "langchain_rag_dataset.txt"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Required dataset file not found: {data_path}")

loader: TextLoader = TextLoader(data_path, encoding="utf-8")
raw_docs: List[Document] = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len
)
chunks: List[Document] = text_splitter.split_documents(raw_docs)

print(f"Total raw documents: {len(raw_docs)}")
print(f"Total chunks created: {len(chunks)}")

assert len(raw_docs) > 0, "Raw documents list must not be empty."
assert len(chunks) >= 3, "Chunks count must be >= 3."


Total raw documents: 1
Total chunks created: 12


## 3. Local Dense Embedding Generation & FAISS Indexing

We instantiate `sentence-transformers/all-MiniLM-L6-v2` locally and index all chunks into a FAISS vector store.

In [3]:
embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
embeddings: HuggingFaceEmbeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name
)

vector_store: FAISS = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"FAISS index created with {vector_store.index.ntotal} vectors.")
assert vector_store.index.ntotal == len(chunks), "Vector store count must match chunk count."


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index created with 12 vectors.


## 4. Configuring the MMR Retriever (`search_type='mmr'`)

We instantiate the retriever with `search_type="mmr"`, requesting `k=3` final documents selected from a candidate pool of `fetch_k=20` with balancing coefficient `lambda_mult=0.5`.

In [4]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

test_query: str = "How does LangChain support agents and memory?"
mmr_docs: List[Document] = mmr_retriever.invoke(test_query)

print(f"Retrieved {len(mmr_docs)} diverse chunks using MMR:")
for idx, doc in enumerate(mmr_docs, start=1):
    print(f"\n[MMR Chunk {idx} | Length: {len(doc.page_content)}]")
    print(doc.page_content.strip())

assert len(mmr_docs) == 3, "MMR retriever must return exactly 3 documents."


Retrieved 3 diverse chunks using MMR:

[MMR Chunk 1 | Length: 215]
history, allowing the agent to recall prior tool outputs and user preferences while executing multi-step reasoning tasks. This composability makes LangChain a primary framework for developing scalable GenAI systems.

[MMR Chunk 2 | Length: 291]
When building conversational applications, memory management is essential. LangChain supports diverse memory abstractions such as ConversationBufferMemory, ChatMessageHistory, and summary memory. These components enable LLMs to maintain state across conversational turns, resolving anaphoric

[MMR Chunk 3 | Length: 73]
model interaction, document indexing, and autonomous agent orchestration.


## 5. Comparative Validation: Standard Similarity vs. MMR Diversification

We evaluate the difference in contextual coverage between standard similarity search and MMR for queries spanning multiple subtopics.

In [5]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

sim_docs: List[Document] = similarity_retriever.invoke(test_query)

print("=== Standard Similarity Top Chunks ===")
for i, d in enumerate(sim_docs, 1):
    print(f"Doc {i}: {d.page_content[:95]}...")

print("\n=== MMR Diversified Chunks ===")
for i, d in enumerate(mmr_docs, 1):
    print(f"Doc {i}: {d.page_content[:95]}...")

assert len(sim_docs) == 3, "Similarity retriever must return 3 docs."
assert len(mmr_docs) == 3, "MMR retriever must return 3 docs."


=== Standard Similarity Top Chunks ===
Doc 1: history, allowing the agent to recall prior tool outputs and user preferences while executing m...
Doc 2: When building conversational applications, memory management is essential. LangChain supports d...
Doc 3: LangChain is an open-source development framework designed to simplify the creation of applicat...

=== MMR Diversified Chunks ===
Doc 1: history, allowing the agent to recall prior tool outputs and user preferences while executing m...
Doc 2: When building conversational applications, memory management is essential. LangChain supports d...
Doc 3: model interaction, document indexing, and autonomous agent orchestration....


## 6. End-to-End RAG Chain Execution

We compose `create_stuff_documents_chain` and `create_retrieval_chain` to pass the MMR-diversified context to the LLM for grounded answer generation.

In [6]:
prompt: ChatPromptTemplate = ChatPromptTemplate.from_template("""
Answer the question based on the context provided:
<context>
{context}
</context>
Question: {input}
""")

groq_key: Optional[str] = os.getenv("GROQ_API_KEY")
if groq_key:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
else:
    from langchain_core.language_models.fake import FakeListLLM
    llm = FakeListLLM(responses=[
        "LangChain supports agents by enabling LLMs to act as decision makers that iteratively reason and invoke tools such as search APIs, SQL executors, and math calculators using ReAct patterns. For conversational memory, it provides abstractions like ConversationBufferMemory and ChatMessageHistory, injecting dialogue state across turns."
    ])

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(mmr_retriever, document_chain)

response: Dict[str, Any] = rag_chain.invoke({"input": test_query})

print("=== Generated RAG Answer ===")
print(response["answer"])

print(f"\nRetrieved context count: {len(response['context'])}")
assert "answer" in response, "Response dictionary must contain 'answer'."
assert len(response["answer"].strip()) > 0, "Answer must not be empty."
assert len(response["context"]) == 3, "Context list must contain 3 documents."


=== Generated RAG Answer ===
**LangChain’s support for agents and memory**

| Feature | How LangChain implements it | What it enables |
|---------|----------------------------|-----------------|
| **Agent‑centric design** | • Agents are built on top of LangChain’s *tool* abstraction, letting them call external APIs, run code, or query databases. <br>• The framework keeps a **history** of every tool invocation and the user’s preferences, so the agent can “recall” what it did before and use that context in subsequent steps. | • Multi‑step reasoning and planning become possible because the agent can refer back to earlier outputs. <br>• The agent can adapt its behavior based on user preferences that were recorded earlier. |
| **Memory abstractions** | • **ConversationBufferMemory** – stores the entire chat history in a buffer that the LLM can read. <br>• **ChatMessageHistory** – a more structured history that can be queried or edited. <br>• **SummaryMemory** – keeps a running summary of th